# E(3)-Equivariance

Symmetry as the only universal prior

**Hero equation:** `phi(g * x) = g * phi(x), for all g in E(3)`

---


## Pipeline

5-stage equivariant pipeline: input graph → encode features & positions → equivariant message passing (rotation invariance preserved) → decode → output. Any rotation/translation of input produces the same rotation/translation of output.

1. Input graph
2. Equivariant encoder
3. E(3)-symmetric message passing
4. Equivariant decoder
5. Output


## Mathematical formulation

### Stage 1 — The Euclidean group E(3)$$E(3) = \{(R, t) : R \in O(3), t \in \mathbb{R}^3\}$$E(3) is the group of all rigid transformations of 3D space — rotations (R ∈ O(3)) and translations (t ∈ ℝ³). Two scans of the same patient taken with different probe orientations differ by an E(3) transformation. Equivariance means the network's output transforms predictably under E(3).**Variables:**- `R`: 3×3 orthogonal rotation matrix (R^T R = I) [dimensionless]- `t`: Translation vector [spatial]- `O(3)`: Group of 3×3 orthogonal matrices [group]---### Stage 2 — Equivariance definition$$\phi(g \cdot x) = g \cdot \phi(x), \quad \forall g \in E(3)$$A function φ is E(3)-equivariant if applying a transformation g to the input and then computing φ is the same as computing φ first and then transforming the output. For medical imaging: rotating the patient's scan before reconstruction gives the same result as reconstructing first and then rotating the volume.**Variables:**- `φ`: Neural network function [operator]- `g`: Element of E(3) (rotation + translation) [operator]- `x`: Input (e.g., 2D particle projections) [varies]---### Stage 3 — Coordinate message passing$$m_{j \to i} = \phi_e(h_i, h_j, \|x_i - x_j\|^2)$$Messages are computed from sender features h_j, receiver features h_i, and the SQUARED distance ‖x_i - x_j‖². Using squared distance (not raw coordinates) ensures rotation invariance of the message. The features h_i, h_j carry rotation-invariant scalar information.**Variables:**- `h_i`: Feature vector at node i (rotation-invariant scalars) [varies]- `x_i`: Position of node i in 3D [spatial]- `m_{j→i}`: Message from node j to node i [varies]- `φ_e`: Edge MLP (message function) [operator]---### Stage 4 — Coordinate update (equivariant)$$x_i^{(l+1)} = x_i^{(l)} + \Pi_{c,\epsilon}\left(\frac{1}{|\mathcal{N}(i)|} \sum_{j \in \mathcal{N}(i)} (x_i - x_j) \cdot \phi_x(m_{j \to i})\right)$$The coordinate update is a sum of (x_i - x_j) weighted by message-derived scalars φ_x(m). Because each (x_i - x_j) is a vector and φ_x outputs a scalar, the sum is a vector that rotates with the input — making the update E(3)-equivariant. TME-FUSE wraps this in Π for the displacement bound.**Variables:**- `𝒩(i)`: Set of neighbors of node i [set]- `φ_x`: Coordinate MLP (outputs scalar weight) [operator]- `Π_{c,ε}`: TME-FUSE norm-clamp operator [operator]---### Stage 5 — Feature update (invariant)$$h_i^{(l+1)} = h_i^{(l)} + \phi_h\left(\sum_{j \in \mathcal{N}(i)} m_{j \to i}\right)$$Feature vectors h are updated by summing messages — a permutation-invariant operation. Because messages themselves are rotation-invariant (computed from squared distances), the feature update is fully E(3)-invariant. This separates the equivariant (coordinate) stream from the invariant (feature) stream — the key architectural insight of EGNN.**Variables:**- `φ_h`: Feature MLP [operator]---

## Math → Code mapping

| Math | Code |
|------|-------|
| `φ_e(h_i, h_j, ‖x_i - x_j‖²)` | `edge_mlp (h_ij @ edge_mlp_W1 ...)` || `φ_x(m)` | `coord_mlp (m @ coord_mlp_W)` || `φ_h(Σ m)` | `feat_mlp (m_sum @ feat_mlp_W1 ...)` || `Σ_{j∈N(i)} (x_i - x_j) · φ_x(m)` | `(diff * w[..., None]).sum(axis=1)` || `Π_{c,ε}(Δx)` | `norm_clamp(delta, c=self.c)` || `h_i^(l+1) = h_i^(l) + φ_h(...)` | `h_new = h + ...` || `g · x` | `x @ R.T (rotation by R)` |

## Runnable code

File: `e3_egnn.py`


In [ ]:
# E(3)-Equivariant Graph Neural Network layer (TME-FUSE flavored)
# Math -> Code mapping:
#   g in E(3)       -> not explicit; equivariance is structural
#   phi_e(...)      -> edge_mlp (features + squared distance -> message)
#   phi_x(...)      -> coord_mlp (message -> scalar weight)
#   phi_h(...)      -> feat_mlp (summed messages -> new features)
#   Pi_{c,eps}(...) -> norm_clamp (coordinate update)
import numpy as np

def norm_clamp(v, c=2.0, eps=1e-8):
    norms = np.linalg.norm(v, axis=-1, keepdims=True)
    return v * np.minimum(1.0, c / (norms + eps))

class EGNNLayer:
    """One layer of E(3)-equivariant graph neural network."""
    def __init__(self, feat_dim=64, c=2.0):
        self.feat_dim = feat_dim
        self.c = c
        # MLPs (simplified as random weight matrices for demo)
        rng = np.random.default_rng(0)
        self.edge_mlp_W1 = rng.standard_normal((2*feat_dim + 1, feat_dim))
        self.edge_mlp_W2 = rng.standard_normal((feat_dim, feat_dim))
        self.coord_mlp_W = rng.standard_normal((feat_dim, 1))
        self.feat_mlp_W1 = rng.standard_normal((feat_dim, feat_dim))
        self.feat_mlp_W2 = rng.standard_normal((feat_dim, feat_dim))

    def forward(self, h, x, adj):
        """
        Args:
            h: (N, feat_dim) node features (rotation-invariant)
            x: (N, 3) node positions
            adj: (N, N) adjacency (1 = edge, 0 = no edge)
        Returns:
            h_new, x_new
        """
        N = h.shape[0]
        # Compute pairwise differences and squared distances
        diff = x[:, None, :] - x[None, :, :]  # (N, N, 3) — x_i - x_j
        d_sq = (diff ** 2).sum(axis=-1, keepdims=True)  # (N, N, 1) — ||x_i - x_j||^2

        # Edge MLP: phi_e(h_i, h_j, d_sq) -> message
        h_ij = np.concatenate([
            np.broadcast_to(h[:, None, :], (N, N, self.feat_dim)),  # h_i
            np.broadcast_to(h[None, :, :], (N, N, self.feat_dim)),  # h_j
            d_sq,
        ], axis=-1)  # (N, N, 2*feat_dim + 1)
        m = np.maximum(0, h_ij @ self.edge_mlp_W1) @ self.edge_mlp_W2  # (N, N, feat_dim)
        m = m * adj[..., None]  # mask to neighbors

        # Coordinate MLP: phi_x(m) -> scalar weight
        w = (m @ self.coord_mlp_W).squeeze(-1)  # (N, N)

        # Coordinate update (EQUIVARIANT — sum of (x_i - x_j) * scalar w)
        delta = (diff * w[..., None]).sum(axis=1)  # (N, 3)
        # Apply norm-clamp (TME-FUSE)
        delta = norm_clamp(delta, c=self.c)
        # Mean over neighbors
        deg = adj.sum(axis=1, keepdims=True)
        delta = delta / np.maximum(deg, 1)
        x_new = x + delta

        # Feature update (INVARIANT — sum of messages)
        m_sum = m.sum(axis=1)  # (N, feat_dim)
        h_new = h + np.maximum(0, m_sum @ self.feat_mlp_W1) @ self.feat_mlp_W2

        return h_new, x_new

# === Demonstration: verify equivariance ===
rng = np.random.default_rng(42)
N, F = 50, 64
h = rng.standard_normal((N, F))  # random features
x = rng.standard_normal((N, 3)) * 5  # random positions
adj = (np.linalg.norm(x[:, None] - x[None, :], axis=-1) < 4.0).astype(float)

layer = EGNNLayer(feat_dim=F, c=2.0)
h_out, x_out = layer.forward(h, x, adj)

# Apply a random rotation R to input
theta = np.pi / 4
R = np.array([[np.cos(theta), -np.sin(theta), 0],
              [np.sin(theta),  np.cos(theta), 0],
              [0, 0, 1]])
x_rot = x @ R.T  # rotate all positions

h_out_rot, x_out_rot = layer.forward(h, x_rot, adj)

# Equivariance check: rotating input then forward == rotating forward output
x_out_expected = x_out @ R.T
equivariance_error = np.linalg.norm(x_out_rot - x_out_expected) / np.linalg.norm(x_out_expected)
print(f"Equivariance error (should be ~0): {equivariance_error:.2e}")
print(f"Max coordinate displacement: {np.linalg.norm(x_out - x, axis=-1).max():.4f} (bound: c=2.0)")


## Observations

**Dimensionality:** 5D

50 nodes in 3D space, one EGNN layer forward pass. Color = per-node displacement magnitude (green = small, red = near bound c). Animation: rotate the input positions continuously; observe the output positions rotate identically. The displacement magnitudes (colors) stay constant — that is equivariance.

- **Canvas type:** rotating-points
- **Points:** 50
- **Animation:** 10s


## Deep thoughts

### ✅ Use when
- Your data has rotational symmetry that a standard CNN would destroy. Discrete 3D CNNs sample on a fixed voxel grid — rotate the input by 45° and you get a different (and wrong) reconstruction. E(3)-equivariant networks give identical reconstructions for any probe orientation.
- You have low data volume (medical imaging is famously data-scarce). Equivariance is a strong inductive bias that effectively multiplies your dataset by |E(3)| = ∞ (every rotation is implicitly in your training set).
- You need to certify that two scans of the same patient give the same diagnosis regardless of how they were acquired. Equivariance is the formal name for this property.

### ⚠️ Avoid when
- Your data has a preferred orientation (e.g., satellite imagery of Earth is always north-up). Forcing equivariance wastes capacity — your network has to learn that the input is rotation-invariant when it is not.
- You need directional features (e.g., 'left ventricle is on the left side of the body'). Equivariant features cannot distinguish left from right — they are rotated to each other.
- Your compute budget is extremely tight. Equivariant layers cost ~2-3× more per parameter than vanilla MLPs due to the message passing overhead.

### 💭 Why
- Equivariance is not just a regularization — it is a mathematical identity. A network that is provably equivariant does not need to 'learn' rotation invariance; it is structurally guaranteed. This is the difference between a network that is robust on average and a network that is robust always.
- E(3)-equivariance is the natural group for 3D physical data, but it is not the only option. SE(3) (rotation + translation, no reflection) is appropriate when the data is chiral (e.g., proteins, DNA). O(3) (rotation only) is appropriate for centered data. The choice of group is a domain knowledge commitment.
- The 2021 EGNN paper showed that this single architectural prior matches or beats far larger non-equivariant networks on molecular dynamics, crystallization, and graph modeling. The 'symmetry as inductive bias' paradigm has been validated in every domain it has been tried in since.


## Comparison with related approaches

### vs Discrete 3D CNN (VoxelNet, etc.)3D CNNs sample on a fixed voxel grid. Rotate the input by 45° and the network sees a completely different input — it must learn rotation invariance from training data. E(3)-equivariant networks give identical outputs for any rotation by construction. Cost: 2-3× compute per parameter.### vs Data augmentationAugmenting training data with random rotations effectively teaches the network rotation invariance — but only on the rotations seen during training. Equivariance is structural: it generalizes to ALL rotations, including ones never seen during training. The two are complementary (use both).### vs Spherical harmonics (Cohen et al. 2018)Spherical harmonic networks (TensorFieldNet, SE(3)-Transformer) achieve equivariance via harmonic bases — more expressive but more expensive. EGNN achieves it via a simpler message-passing form that is empirically competitive on most tasks at 1/10 the compute.### vs Standard GNN (GAT, GCN)Standard GNNs are permutation-equivariant (node order) but not rotation-equivariant (positions). They treat node positions as arbitrary features. EGNN treats positions as first-class equivariant objects — the difference is whether you can take the same scan at two orientations and get the same answer.

## Futuristic possibilities

### 2026-27 — Group generalizationE(3)-equivariance extends to non-Euclidean groups: SE(2) for satellite imagery, Lorentz for cosmology, Weyl for crystallography, permutation-equivariance for sets. The AISciImg360 adapter API exposes group choice as a per-modality parameter.### 2028-30 — Equivariant diffusion modelsEquivariant diffusion models (EDM, e3-equivalent of DDPM) become the standard for de novo molecular generation. Drug discovery moves from virtual screening to virtual generation, with equivariance guaranteeing that generated molecules respect physical symmetries.### 2030-33 — Equivariant world modelsRobotics foundation models are E(3)-equivariant by design. A robot that learns to grasp a cup does not need to re-learn the grasp when the cup is rotated 90°. Equivariance collapses the state space by |E(3)| = ∞ and makes reinforcement learning tractable in continuous control.### 2033+ — Symmetry as the universal priorThe history of deep learning is the history of inductive biases: CNNs (translation equivariance), transformers (permutation equivariance), EGNNs (rotation+translation equivariance). By 2033, every domain-specific architecture is replaced by a universal equivariant foundation model + thin adapter.

## Attribution

Inspired by Victor Garcia Satorras' EGNN paper and the TME-FUSE framework's norm-clamped extension.

> Satorras, V.G., Hoogeboom, E., Welling, M. (2021). E(n) Equivariant Graph Neural Networks. ICML 2021.

---

*Generated by OSIRIS / AISciImg360 concept card → notebook converter.*
*Open the [IDE studio](/ide) to run this code in your browser via Pyodide.*
